In [1]:
%pip install -q transformers accelerate sentence-transformers tqdm

Note: you may need to restart the kernel to use updated packages.


## Импорты и параметры

In [8]:
import os

os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from implicit.gpu.als import AlternatingLeastSquares
    print('implicit: GPU')
except ImportError:
    from implicit.als import AlternatingLeastSquares
    print('implicit: CPU')

import warnings
warnings.filterwarnings('ignore')

# ===== Константы (совпадают с als_catboost.ipynb) =====
DATA_DIR = Path('../../data/children_products')
CLEANED_CSV = DATA_DIR / 'clildren_product_cleaned.csv'
RAW_CSV = DATA_DIR / '!03&04_17_VSE.csv'

MIN_INTERACTIONS = 5
TRAIN_SPLIT = 0.8

N_EVAL_USERS = 300
K_RECOMMEND = 30           # сколько рекомендаций выдавать у LLM
TOP_M_PER_QUERY = 3        # сколько SKU на каждую строку от LLM
FINAL_K = 20               # финальный top-K для метрик
MAX_HISTORY_ITEMS = 30     # обрезаем хвост истории пользователя

EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
# Не-gated зеркало Llama 3.1 8B Instruct. Если залогинены в HF и приняли
# лицензию официальной модели — замените на 'meta-llama/Meta-Llama-3.1-8B-Instruct'.
LLM_MODEL = 'NousResearch/Meta-Llama-3.1-8B-Instruct'

SKU_EMB_CACHE = DATA_DIR / 'sku_embeddings.npy'
# Отдельный файл, чтобы не затирать кэш оригинального Ollama-ноутбука
LLM_CACHE = DATA_DIR / 'llm_recommendations_cache_transformers.json'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


implicit: GPU


In [9]:
df = pd.read_csv(CLEANED_CSV)
print(f'Исходный датасет: {df.shape}')

raw = pd.read_csv(
    RAW_CSV,
    sep=';', encoding='cp1251',
    usecols=['ID_SKU', 'Номенклатура', 'Группа2', 'Группа3', 'Тип'],
    dtype=str, low_memory=False
)
raw = raw.dropna(subset=['ID_SKU']).drop_duplicates('ID_SKU')
print(f'Уникальных SKU в сыром CSV: {len(raw):,}')

df = df.merge(raw[['ID_SKU', 'Номенклатура']], on='ID_SKU', how='left')
missing = df['Номенклатура'].isna().mean()
print(f'Доля строк без Номенклатуры: {missing:.2%}')

fallback = df['Группа3'].fillna('') + ' / ' + df['Тип'].fillna('')
df['Номенклатура'] = df['Номенклатура'].fillna(fallback)
df.head(3)

Исходный датасет: (610913, 16)
Уникальных SKU в сыром CSV: 85,380
Доля строк без Номенклатуры: 0.00%


,Дата,НомерЗаказаНаСайте,МетодДоставки,Группа2,Группа3,Тип,Отменено,Количество,Цена,Статус,Гео,Маржа,СуммаУслуг,Телефон_new,ID_SKU,МетодДоставки_Групп,Номенклатура
0,2017-03-01 11:41:00,3998972_TR,Курьерская,КРУПНОГАБАРИТНЫЙ ТОВАР,КОЛЯСКИ,КГТ,Нет,1,680.0,Возврат,Москва,508.0,0,55574854-48574951555577,ID9010020114553,Курьерская,"LEADER KIDS, МУФТА на ручку коляски, (беж),"
1,2017-03-01 12:22:00,3999117_TR,Магазины,"ТЕКСТИЛЬ, ТРИКОТАЖ",ОДЕЖДА ДЛЯ НОВОРОЖДЕННЫХ (0-2 лет),ОДЕЖДА,Нет,1,379.0,Доставлен,Регионы,169.2,0,55575453-56535648535679,IDL00028974351,Магазины,"GAMEX, БОДИ кор. рук. Roza, (бел/сер), р. 86, ..."
2,2017-03-01 12:31:00,3999122_TR,Магазины,ИГРУШКИ,ИГРУШКИ ДЛЯ ДЕВОЧЕК,ИГРУШКИ,Нет,1,3325.0,Доставлен,Регионы,2176.0,0,55574950-57515657535772,IDL00038573351,Магазины,"ИГРУША, ХОЛОДИЛЬНИК (на бат), (29,5*19*45,5 см)"


In [10]:
df_filtered = df[(df['Статус'] == 'Доставлен') & (df['Отменено'] == 'Нет')].copy()
df_filtered = df_filtered.dropna(subset=['Телефон_new', 'ID_SKU', 'Дата'])
df_filtered['Дата'] = pd.to_datetime(df_filtered['Дата'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['Дата'])

for _ in range(5):
    uc = df_filtered.groupby('Телефон_new').size()
    ic = df_filtered.groupby('ID_SKU').size()
    active_users = uc[uc >= MIN_INTERACTIONS].index
    active_items = ic[ic >= MIN_INTERACTIONS].index
    before = len(df_filtered)
    df_filtered = df_filtered[
        df_filtered['Телефон_new'].isin(active_users) &
        df_filtered['ID_SKU'].isin(active_items)
    ]
    if len(df_filtered) == before:
        break

user_enc, item_enc = LabelEncoder(), LabelEncoder()
df_filtered['user_id'] = user_enc.fit_transform(df_filtered['Телефон_new'])
df_filtered['item_id'] = item_enc.fit_transform(df_filtered['ID_SKU'])
n_users, n_items = len(user_enc.classes_), len(item_enc.classes_)
print(f'Пользователей: {n_users:,}  Товаров: {n_items:,}  Взаимодействий: {len(df_filtered):,}')

# Temporal split
df_filtered = df_filtered.sort_values('Дата')
split_date = df_filtered['Дата'].quantile(TRAIN_SPLIT)
train_data = df_filtered[df_filtered['Дата'] < split_date].copy()
test_data = df_filtered[df_filtered['Дата'] >= split_date].copy()
print(f'Split date: {split_date}')
print(f'Train: {len(train_data):,}  Test: {len(test_data):,}')

# Sparse матрицы для ALS
train_int = train_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
test_int = test_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
train_matrix = csr_matrix(
    (train_int['count'].values, (train_int['user_id'].values, train_int['item_id'].values)),
    shape=(n_users, n_items)
)

test_user_items = test_int.groupby('user_id')['item_id'].apply(list).to_dict()
train_user_set = set(train_int['user_id'].unique())
warm_eval_users = [u for u in test_user_items if u in train_user_set]
print(f'Warm-start test users: {len(warm_eval_users):,}')

# Подвыборка для LLM
rng = np.random.default_rng(RANDOM_SEED)
sample_users = rng.choice(warm_eval_users, size=min(N_EVAL_USERS, len(warm_eval_users)), replace=False)
sample_users = sorted(map(int, sample_users))
print(f'Подвыборка для LLM: {len(sample_users)} пользователей')

Пользователей: 17,442  Товаров: 8,488  Взаимодействий: 227,170
Split date: 2017-04-18 05:36:00
Train: 181,716  Test: 45,454
Warm-start test users: 4,196
Подвыборка для LLM: 300 пользователей


Подготовка эмбедингов

In [11]:
# Один канонический текст на item_id (мода по Номенклатуре в train)
item_name_map = (
    df_filtered.groupby('item_id')['Номенклатура']
    .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
)
item_names = [item_name_map.loc[i] for i in range(n_items)]
print(f'Примеры названий: {item_names[:3]}')

if SKU_EMB_CACHE.exists():
    sku_emb = np.load(SKU_EMB_CACHE)
    print(f'Загружены эмбеддинги из кэша: {sku_emb.shape}')
    if sku_emb.shape[0] != n_items:
        print('Размер кэша не совпадает — пересчитываем')
        sku_emb = None
else:
    sku_emb = None

if sku_emb is None:
    encoder = SentenceTransformer(EMBED_MODEL)
    sku_emb = encoder.encode(item_names, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    np.save(SKU_EMB_CACHE, sku_emb)
    print(f'Эмбеддинги сохранены: {sku_emb.shape}')

knn = NearestNeighbors(metric='cosine', algorithm='brute').fit(sku_emb)

Примеры названий: ['ПАЗЛ КТО где живет', 'МАШИНА легковая Нордик ', 'УШАСТЫЙ НЯНЬ, ГЕЛЬ для посуды с ромашкой, (500 мл)']
Загружены эмбеддинги из кэша: (8488, 768)


Подготовка истории

In [12]:

train_sorted = train_data.sort_values('Дата')
user_history_cache = {}
for uid, g in train_sorted.groupby('user_id'):
    user_history_cache[int(uid)] = list(zip(g['Дата'].dt.date.astype(str), g['Номенклатура']))
user_train_items_cache = {
    int(uid): set(g['item_id'].tolist())
    for uid, g in train_sorted.groupby('user_id')
}

def build_user_history(user_id: int, max_items: int = MAX_HISTORY_ITEMS) -> str:
    items = user_history_cache.get(user_id, [])
    tail = items[-max_items:]
    return '\n'.join(f'- {date}: {name}' for date, name in tail)

print(build_user_history(sample_users[0]))

- 2017-03-11: ГЕРБЕР, ПЮРЕ груша-сливки, (125г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ банан-творог, (125 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ персик-творог, (125 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ банан-сливки, (125 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ овощи тушеные с телятиной, (130 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ банан-творог, (125 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ овощи тушеные с телятиной, (130 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ груша-сливки, (125г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ банан-сливки, (125 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ персик-творог, (125 г)
- 2017-03-12: СИМИЛАК ГА 2, ЗАМЕНИТЕЛЬ молока 6-12 мес., (400 г)
- 2017-03-13: НЕСТЛЕ, КАША б/мол., гречневая, с черносливом, (200 г)
- 2017-03-13: НЕСТЛЕ, КАША б/мол., кукурузная, (200 г)
- 2017-03-13: НЕСТЛЕ, КАША б/мол., 5 злаков, липовый цвет, (200 г)
- 2017-03-13: НЕСТЛЕ, КАША б/мол., гречневая, (200 г)
- 2017-03-13: НЕСТЛЕ, КАША б/мол., 5 злаков, (200 г)
- 2017-03-29: ГЕРБЕР, ПЮРЕ груша-сливки, (125г)
- 2017-03-29: ГЕРБЕР, ПЮРЕ банан-творог, (125 г)
- 2017-03-29: ГЕРБЕР, ПЮ

In [ ]:
SYSTEM_PROMPT = (
    'Ты — рекомендательная система детского интернет-магазина. '
    'На основе истории покупок пользователя предложи список новых товаров, '
    'которые он вероятно купит в ближайшее время. '
    'Ориентируйся на возраст ребёнка, предпочитаемые категории, бренды, ценовой сегмент. '
    'Не повторяй товары из истории. '
    'Отвечай СТРОГО в формате JSON: {"recommendations": ["название1", "название2", ...]}. '
    'Каждое название должно быть в формате "БРЕНД, ТИП, характеристики" (как в истории).'
)

# ===== Загрузка модели (однократно) =====
if torch.cuda.is_available():
    llm_device = 'cuda'
    llm_dtype = torch.bfloat16
    device_map = 'auto'
elif torch.backends.mps.is_available():
    llm_device = 'mps'
    llm_dtype = torch.float16
    device_map = None
else:
    llm_device = 'cpu'
    llm_dtype = torch.float32
    device_map = None
    print('[warn] GPU не найден — Llama 3.1 8B на CPU будет очень медленно.')

print(f'Loading {LLM_MODEL} on {llm_device} (dtype={llm_dtype})...')
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=llm_dtype,
    device_map=device_map,
    low_cpu_mem_usage=True,
)
if device_map is None:
    model = model.to(llm_device)
model.eval()

_eot_id = tokenizer.convert_tokens_to_ids('<|eot_id|>')
_raw_terms = [tokenizer.eos_token_id, _eot_id]
LLM_TERMINATORS = sorted({
    int(tid) for tid in _raw_terms
    if isinstance(tid, int) and tid is not None and tid >= 0
})
print(f'Model loaded. Terminators: {LLM_TERMINATORS}')


def _parse_json_array(raw_text: str) -> list:
    try:
        obj = json.loads(raw_text)
        if isinstance(obj, dict):
            for key in ('recommendations', 'items', 'products', 'list'):
                if key in obj and isinstance(obj[key], list):
                    return [str(x) for x in obj[key] if str(x).strip()]
            # Возможно dict с одним списком
            for v in obj.values():
                if isinstance(v, list):
                    return [str(x) for x in v if str(x).strip()]
        if isinstance(obj, list):
            return [str(x) for x in obj if str(x).strip()]
    except Exception:
        # Фолбэк: попытка вытащить массив регуляркой
        m = re.search(r'\[[\s\S]*\]', raw_text)
        if m:
            try:
                arr = json.loads(m.group(0))
                if isinstance(arr, list):
                    return [str(x) for x in arr if str(x).strip()]
            except Exception:
                pass
    return []


def ask_llm(history_str: str, k: int = K_RECOMMEND, retries: int = 1) -> list:
    user_prompt = (
        f'История покупок пользователя:\n{history_str}\n\n'
        f'Предложи {k} новых товаров в JSON: '
        '{"recommendations": ["..."]}'
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ]
    # tokenize=False → получаем plain text, затем прогоняем через tokenizer()
    # чтобы получить и input_ids, и attention_mask (которых нет при return_tensors напрямую).
    prompt_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[-1]

    temp = 0.3
    for attempt in range(retries + 1):
        try:
            with torch.inference_mode():
                out = model.generate(
                    **inputs,
                    max_new_tokens=1500,
                    do_sample=True,
                    temperature=temp,
                    top_p=0.9,
                    top_k=50,  # стабилизирует multinomial на MPS
                    eos_token_id=LLM_TERMINATORS,
                    pad_token_id=tokenizer.pad_token_id,
                )
            new_tokens = out[0, input_len:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True)
            parsed = _parse_json_array(text)
            if parsed:
                return parsed[:k]
        except Exception as e:
            print(f'[warn] LLM error (attempt {attempt}): {e}')
        temp = 0.1
    return []


# Sanity check на одном пользователе
_test_user = sample_users[0]
_test_hist = build_user_history(_test_user)
print('=== История ===')
print(_test_hist[:500], '...' if len(_test_hist) > 500 else '')
print('\n=== Ответ LLM ===')
_test_out = ask_llm(_test_hist, k=10)
for i, x in enumerate(_test_out, 1):
    print(f'{i}. {x}')

Loading NousResearch/Meta-Llama-3.1-8B-Instruct on mps (dtype=torch.float16)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded. Terminators: [128009]
=== История ===
- 2017-03-11: ГЕРБЕР, ПЮРЕ груша-сливки, (125г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ банан-творог, (125 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ персик-творог, (125 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ банан-сливки, (125 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ овощи тушеные с телятиной, (130 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ банан-творог, (125 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ овощи тушеные с телятиной, (130 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ груша-сливки, (125г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ банан-сливки, (125 г)
- 2017-03-11: ГЕРБЕР, ПЮРЕ персик- ...

=== Ответ LLM ===


KeyboardInterrupt: 

In [ ]:
encoder = SentenceTransformer(EMBED_MODEL)

def llm_to_skus(
    llm_outputs: list,
    user_train_items: set,
    final_k: int = FINAL_K,
    m_per_query: int = TOP_M_PER_QUERY,
) -> list:
    if not llm_outputs:
        return []
    q_emb = encoder.encode(llm_outputs, batch_size=32, show_progress_bar=False, convert_to_numpy=True)
    _, idx = knn.kneighbors(q_emb, n_neighbors=m_per_query)
    seen, ranked = set(), []
    for row in idx:
        for cand in row:
            c = int(cand)
            if c in user_train_items or c in seen:
                continue
            seen.add(c)
            ranked.append(c)
            if len(ranked) >= final_k:
                return ranked
    return ranked

_test_skus = llm_to_skus(_test_out, user_train_items_cache.get(_test_user, set()))
for sku_id in _test_skus[:10]:
    print(f'  item_id={sku_id:5d}  {item_names[sku_id][:90]}')

Вычисление

In [ ]:
# Загружаем кэш если есть
llm_recs = {}
llm_raw = {}
if LLM_CACHE.exists():
    with open(LLM_CACHE, 'r', encoding='utf-8') as f:
        cache = json.load(f)
    llm_recs = {int(k): v for k, v in cache.get('recs', {}).items()}
    llm_raw = {int(k): v for k, v in cache.get('raw', {}).items()}
    print(f'Восстановлено из кэша: {len(llm_recs)} пользователей')

to_process = [u for u in sample_users if u not in llm_recs]
print(f'К обработке: {len(to_process)} пользователей')

save_every = 20
for i, uid in enumerate(tqdm(to_process, desc='LLM inference')):
    hist = build_user_history(uid)
    raw_out = ask_llm(hist, k=K_RECOMMEND)
    llm_raw[uid] = raw_out
    skus = llm_to_skus(raw_out, user_train_items_cache.get(uid, set()))
    llm_recs[uid] = skus
    if (i + 1) % save_every == 0:
        with open(LLM_CACHE, 'w', encoding='utf-8') as f:
            json.dump({'recs': {str(k): v for k, v in llm_recs.items()},
                       'raw': {str(k): v for k, v in llm_raw.items()}}, f, ensure_ascii=False)

with open(LLM_CACHE, 'w', encoding='utf-8') as f:
    json.dump({'recs': {str(k): v for k, v in llm_recs.items()},
               'raw': {str(k): v for k, v in llm_raw.items()}}, f, ensure_ascii=False)

non_empty = sum(1 for v in llm_recs.values() if v)
print(f'Готово. Непустых рекомендаций: {non_empty}/{len(llm_recs)}')

In [ ]:
def precision_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(rec_k) if rec_k else 0.0

def recall_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(set(relevant)) if relevant else 0.0

def map_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    score, hits = 0.0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(relevant), k)

def ndcg_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

K_VALUES = [5, 10, 20]

def aggregate(recs_by_user: dict, users: list) -> dict:
    out = {k: {m: [] for m in ['precision', 'recall', 'map', 'ndcg']} for k in K_VALUES}
    for u in users:
        rel = test_user_items.get(u, [])
        if not rel:
            continue
        rec = recs_by_user.get(u, [])
        for k in K_VALUES:
            out[k]['precision'].append(precision_at_k(rec, rel, k))
            out[k]['recall'].append(recall_at_k(rec, rel, k))
            out[k]['map'].append(map_at_k(rec, rel, k))
            out[k]['ndcg'].append(ndcg_at_k(rec, rel, k))
    return {k: {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()} for k, metrics in out.items()}

llm_results = aggregate(llm_recs, sample_users)
print('LLM zero-shot метрики:')
for k in K_VALUES:
    print(f'  @{k}: ', {m: f'{v:.4f}' for m, v in llm_results[k].items()})

In [ ]:
pop_series = train_int.groupby('item_id')['count'].sum()
item_popularity = np.zeros(n_items, dtype=np.float32)
for iid, c in pop_series.items():
    item_popularity[int(iid)] = c
pop_log_max = float(np.log1p(item_popularity.max()))
print(f'Популярность: min={item_popularity.min():.0f}, median={np.median(item_popularity):.0f}, max={item_popularity.max():.0f}')


cat_map = (
    train_data.groupby('item_id')['Группа3']
    .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else 'UNK')
)
item_cat_arr = np.array(
    [cat_map.get(i, 'UNK') for i in range(n_items)], dtype=object
)
print(f'Уникальных категорий Группа3: {len(set(item_cat_arr))}')


user_cat_dist = {}
for uid, g in train_data.groupby('user_id'):
    vc = g['Группа3'].value_counts(normalize=True)
    user_cat_dist[int(uid)] = vc.to_dict()
print(f'Профилей пользователей: {len(user_cat_dist):,}')


_u = sample_users[0]
print(f'\nКатегории пользователя {_u}:')
for cat, share in sorted(user_cat_dist[_u].items(), key=lambda x: -x[1])[:5]:
    print(f'  {share:.2%}  {cat}')

v2 с взвешенным категориями

In [ ]:
def llm_to_skus_v2(
    user_id: int,
    llm_outputs: list,
    user_train_items: set,
    final_k: int = FINAL_K,
    m_per_query: int = 10,
    pop_weight: float = 0.3,
    cat_bonus: float = 0.5,
    out_cat_penalty: float = 0.3,
) -> list:
    if not llm_outputs:
        return []
    q_emb = encoder.encode(llm_outputs, batch_size=32, show_progress_bar=False, convert_to_numpy=True)
    distances, idx = knn.kneighbors(q_emb, n_neighbors=m_per_query)
    sims = 1.0 - distances  # cosine similarity

    # popularity: log-нормированная в [0, 1]
    pop_norm = np.log1p(item_popularity[idx]) / (pop_log_max + 1e-9)

    # category factor: буст если категория в истории юзера, штраф иначе
    user_cats = user_cat_dist.get(int(user_id), {})
    cats_of_cand = item_cat_arr[idx]  # shape (Q, M)
    cat_share = np.vectorize(lambda c: user_cats.get(c, 0.0))(cats_of_cand)
    cat_factor = np.where(cat_share > 0, 1.0 + cat_bonus * cat_share, 1.0 - out_cat_penalty)

    scores = sims * (1.0 + pop_weight * pop_norm) * cat_factor

    # Flatten + sort descending
    flat_iids = idx.flatten()
    flat_scores = scores.flatten()
    order = np.argsort(-flat_scores)

    seen, ranked = set(), []
    for o in order:
        c = int(flat_iids[o])
        if c in user_train_items or c in seen:
            continue
        seen.add(c)
        ranked.append(c)
        if len(ranked) >= final_k:
            break
    return ranked

In [ ]:
llm_recs_v2 = {}
for uid in tqdm(sample_users, desc='retrieval v2'):
    raw = llm_raw.get(int(uid), [])
    user_train = user_train_items_cache.get(int(uid), set())
    llm_recs_v2[int(uid)] = llm_to_skus_v2(uid, raw, user_train)

non_empty = sum(1 for v in llm_recs_v2.values() if v)
print(f'v2 непустых: {non_empty}/{len(llm_recs_v2)}')

llm_results_v2 = aggregate(llm_recs_v2, sample_users)
print('\nLLM zero-shot v2 метрики:')
for k in K_VALUES:
    print(f'  @{k}: ', {m: f"{v:.4f}" for m, v in llm_results_v2[k].items()})

In [ ]:
from implicit.als import AlternatingLeastSquares

als = AlternatingLeastSquares(
    factors=50, regularization=0.01, iterations=15,
    calculate_training_loss=True, random_state=42
)
als.fit(train_matrix)

als_recs = {}
for uid in tqdm(sample_users, desc='ALS recommend'):
    ids, _ = als.recommend(
        uid, train_matrix[uid],
        N=FINAL_K, filter_already_liked_items=True
    )
    als_recs[int(uid)] = [int(x) for x in ids]

als_results = aggregate(als_recs, sample_users)
print('ALS метрики (та же подвыборка):')
for k in K_VALUES:
    print(f'  @{k}: ', {m: f'{v:.4f}' for m, v in als_results[k].items()})

In [ ]:
rows = []
for k in K_VALUES:
    for m in ['precision', 'recall', 'map', 'ndcg']:
        rows.append({
            'metric': f'{m.upper()}@{k}',
            'LLM v1': llm_results[k][m],
            'LLM v2': llm_results_v2[k][m],
            'ALS': als_results[k][m],
        })
comp_df = pd.DataFrame(rows)
comp_df['Δ v2 vs v1'] = (comp_df['LLM v2'] - comp_df['LLM v1']) / (comp_df['LLM v1'] + 1e-10) * 100
comp_df['Δ v2 vs ALS'] = (comp_df['LLM v2'] - comp_df['ALS']) / (comp_df['ALS'] + 1e-10) * 100

print(f"{'Метрика':<15} {'LLM v1':>10} {'LLM v2':>10} {'ALS':>10} {'Δ v2/v1':>10} {'Δ v2/ALS':>10}")
print('-' * 70)
for _, r in comp_df.iterrows():
    print(f"{r['metric']:<15} {r['LLM v1']:>10.4f} {r['LLM v2']:>10.4f} {r['ALS']:>10.4f} "
          f"{r['Δ v2 vs v1']:>+9.1f}% {r['Δ v2 vs ALS']:>+9.1f}%")
    if r['metric'].endswith('@5') and 'NDCG' in r['metric']:
        print()
    elif r['metric'].endswith('@10') and 'NDCG' in r['metric']:
        print()
comp_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, metric in zip(axes.flat, ['precision', 'recall', 'map', 'ndcg']):
    v1 = [llm_results[k][metric] for k in K_VALUES]
    v2 = [llm_results_v2[k][metric] for k in K_VALUES]
    als_v = [als_results[k][metric] for k in K_VALUES]
    ax.plot(K_VALUES, v1, marker='s', linewidth=2, label='LLM v1 (чистый cosine)')
    ax.plot(K_VALUES, v2, marker='^', linewidth=2, label='LLM v2 (pop + cat)')
    ax.plot(K_VALUES, als_v, marker='o', linewidth=2, label='ALS')
    ax.set_xlabel('K')
    ax.set_ylabel(f'{metric.upper()}@K')
    ax.set_title(f'{metric.upper()}@K')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle(f'LLM v1 vs LLM v2 vs ALS (n={len(sample_users)} пользователей)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
example_users = rng.choice(sample_users, size=3, replace=False)
for u in example_users:
    u = int(u)
    print('=' * 80)
    print(f'USER {u}')
    hist = user_history_cache.get(u, [])[-5:]
    print('\nПоследние покупки (train):')
    for date, name in hist:
        print(f'  {date}: {name}')
    print('\nСырые рекомендации LLM (первые 5):')
    for x in llm_raw.get(u, [])[:5]:
        print(f'  • {x}')
    print('\nРекомендации после retrieval (первые 5):')
    for sid in llm_recs.get(u, [])[:5]:
        print(f'  • [{sid}] {item_names[sid][:90]}')
    print('\nПокупки в test:')
    for sid in test_user_items.get(u, [])[:5]:
        print(f'  • [{sid}] {item_names[sid][:90]}')
    print()